## Register the Vitessce notebook kernel

After creating the Vitessce environment, VS Code or Jupyter may not immediately show it in the kernel picker. If you cannot select `targeted-transcriptomics-training-vitessce`, run the following once from a terminal.

On macOS, Linux, or WSL, from the repository root:

```bash
cd vitessce
source .venv/bin/activate
python -m ipykernel install --user \
  --name targeted-transcriptomics-training-vitessce \
  --display-name targeted-transcriptomics-training-vitessce
```

On Windows PowerShell, from the repository root:

```powershell
cd vitessce
. .\.venv\Scripts\Activate.ps1
python -m ipykernel install --user `
  --name targeted-transcriptomics-training-vitessce `
  --display-name targeted-transcriptomics-training-vitessce
```

Then reopen the notebook or refresh the kernel picker and select `targeted-transcriptomics-training-vitessce`.

## MACSima example

In [ ]:
import os

import sys
import tempfile
from pathlib import Path

import harpy as hp
import harpy_vitessce as hpv
from IPython.display import HTML, display


if sys.platform == "win32":
    # Force short paths to avoid running into Windows limitation with path > 260 characters.
    temp_dir = "c:\\tmp"
    course_data_cache_path = "c:\\hp_cache"
    os.makedirs(temp_dir, exist_ok=True)
else:
    temp_dir = None  # use OS default
    course_data_cache_path = None  # use OS default

output_dir = Path(tempfile.TemporaryDirectory(dir=temp_dir, prefix="macsima_").name)

sdata = hp.datasets.macsima_colorectal_carcinoma_course(
    checkpoint="checkpoint_2",
    path=course_data_cache_path,
)

WARNING  Module 'bioio' is not installed. Install it with `pip install bioio` to use `harpy.io.macsima`.           
WARNING  Module 'bioio-ome-tiff' is not installed. Install it with `pip install bioio-ome-tiff` to use             
         `harpy.io.macsima`.                                                                                       


no parent found for <ome_zarr.reader.Label object at 0x162c545f0>: None
no parent found for <ome_zarr.reader.Label object at 0x162d62a50>: None
no parent found for <ome_zarr.reader.Label object at 0x162d62540>: None
no parent found for <ome_zarr.reader.Label object at 0x162d62750>: None


In [2]:
from spatialdata import read_zarr

sdata.write(output_dir / "sdata.zarr", overwrite=True)
sdata = read_zarr(sdata.path)

/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 auto

Lets start simple:

In [3]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    channels=["DAPI"],
    palette=["#00C2FF"],
    to_coordinate_system="global_1_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:28:04.757 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.


In [4]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    channels=["DAPI", "Bcl 2"],
    palette=["#00C2FF", "#FF1744"],
    channel_windows=[(0, 45000), (141, 231)],
    to_coordinate_system="global_1_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:28:16.762 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.


In [5]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    channels=["DAPI", "Bcl 2"],
    zoom=0.3,
    center=(255, 255),
    palette=["#00C2FF", "#FF1744"],
    channel_windows=[(0, 45000), (141, 231)],
    to_coordinate_system="global_1_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

Now add a segmentation mask:

In [6]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    labels_name="labels_cells_instanseg",
    channels=["DAPI"],
    to_coordinate_system="global_1_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:28:42.897 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:28:42.901 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:154 - No palette provided and one channel selected; rendering channel in white (#FFFFFF).


In [7]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    labels_name="labels_cells_instanseg",
    segmentation_filled=False,
    segmentation_stroke_width=0.01,
    segmentation_color="#00E5A8",  # be carefull -> when setting segmentation color, and also a cluster key, the segmentation color will overwrite the cluster key color
    channels=["DAPI"],
    to_coordinate_system="global_1_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:28:54.933 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:28:54.936 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:154 - No palette provided and one channel selected; rendering channel in white (#FFFFFF).


Add a cluster key:

In [8]:
sdata["table_cells_z_scored_filterd"].obs.head()

,cell_ID,fov_labels,perimeter,eccentricity,convex_area,area,filter_on_size,has_nucleus,leiden,shapeSize
cells,,,,,,,,,,
105_labels_cells_instanseg_a57cae0b,105,labels_cells_instanseg,22.182498,0.639923,30.8652,29.1312,False,True,8,1008.0
133_labels_cells_instanseg_a57cae0b,133,labels_cells_instanseg,30.026661,0.848078,43.2633,40.1421,False,True,2,1389.0
141_labels_cells_instanseg_a57cae0b,141,labels_cells_instanseg,22.634163,0.828831,25.4609,23.4957,False,True,1,813.0
145_labels_cells_instanseg_a57cae0b,145,labels_cells_instanseg,23.484163,0.618995,27.9752,25.5765,False,False,1,885.0
149_labels_cells_instanseg_a57cae0b,149,labels_cells_instanseg,22.762914,0.834278,27.0504,25.0563,False,True,5,867.0


In [9]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    labels_name="labels_cells_instanseg",
    table_name="table_cells_z_scored_filterd",
    channels=["DAPI"],
    cluster_key="leiden",
    embedding_key="X_umap",
    to_coordinate_system="global_1_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:29:05.608 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:29:05.610 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:154 - No palette provided and one channel selected; rendering channel in white (#FFFFFF).


Visualize the heatmap and a marker list

In [10]:
vc = hpv.proteomics_from_spatialdata(
    sdata.path,
    image_name="REAscreen_IO_CRC",
    labels_name="labels_cells_instanseg",
    table_name="table_cells_z_scored_filterd",
    channels=["DAPI"],
    cluster_key="leiden",
    embedding_key="X_umap",
    visualize_feature_matrix=True,
    visualize_heatmap=True,
    to_coordinate_system="global_1",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

# or here:
# vc.widget()

2026-06-14 21:29:18.087 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:29:18.091 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:154 - No palette provided and one channel selected; rendering channel in white (#FFFFFF).


## Xenium example

In [12]:
from summer_school_datasets import xenium_human_ovarian_cancer_course

sdata = xenium_human_ovarian_cancer_course(
    checkpoint="checkpoint_2",
    path=course_data_cache_path,
)

output_dir = Path(tempfile.TemporaryDirectory(dir=temp_dir, prefix="xenium_").name)

sdata.write(output_dir / "sdata.zarr", overwrite=True)

sdata = read_zarr(sdata.path)

no parent found for <ome_zarr.reader.Label object at 0x163b90800>: None
no parent found for <ome_zarr.reader.Label object at 0x162bb7830>: None
no parent found for <ome_zarr.reader.Label object at 0x1653f5820>: None
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 auto

In [13]:
vc = hpv.imgbased_transcriptomics_from_spatialdata(
    sdata.path,
    image_name="clahe",
    channels=[0, 1, 2, 3],
    to_coordinate_system="global_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:31:03.013 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:31:03.016 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:158 - No palette provided and 4 channels selected; rendering with the default channel palette.


In [14]:
sdata["table_transcriptomics_preprocessed"].obs.head()

,cell_ID,fov_labels,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_2_genes,pct_counts_in_top_5_genes,n_counts,shapeSize,shapeSize_microns,leiden
cells,,,,,,,,,,,,
1039_nucleus_segmentation_mask_efa3e768,1039,nucleus_segmentation_mask,168,5.129899,218,5.389072,7.339450,12.844037,218,2289.0,103.362663,4
1071_nucleus_segmentation_mask_efa3e768,1071,nucleus_segmentation_mask,312,5.746203,414,6.028279,3.864734,7.487923,414,1884.0,85.074379,5
1135_nucleus_segmentation_mask_efa3e768,1135,nucleus_segmentation_mask,220,5.398163,305,5.723585,5.245902,10.819672,305,2185.0,98.666412,4
1167_nucleus_segmentation_mask_efa3e768,1167,nucleus_segmentation_mask,190,5.252273,239,5.480639,9.623431,14.644351,239,2062.0,93.112190,6
1199_nucleus_segmentation_mask_efa3e768,1199,nucleus_segmentation_mask,106,4.672829,118,4.779123,5.932203,11.864407,118,1027.0,46.375469,2


In [16]:
vc = hpv.imgbased_transcriptomics_from_spatialdata(
    sdata.path,
    image_name="clahe",
    labels_name="nucleus_segmentation_mask",
    segmentation_filled=False,
    segmentation_stroke_width=0.025,
    segmentation_color="#00C2FF",  # segmentation color seems to be ignored if labels element is multiscale.
    channels=[0, 1, 2, 3],
    to_coordinate_system="global_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:31:24.795 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:31:24.798 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:158 - No palette provided and 4 channels selected; rendering with the default channel palette.


In [52]:
[*sdata.tables]

['table_global', 'table_transcriptomics', 'table_transcriptomics_preprocessed']

In [17]:
vc = hpv.imgbased_transcriptomics_from_spatialdata(
    sdata.path,
    image_name="clahe",
    labels_name="nucleus_segmentation_mask",
    table_name="table_transcriptomics_preprocessed",
    channels=[0, 1, 2, 3],
    cluster_key="leiden",
    embedding_key="X_umap",
    to_coordinate_system="global_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:31:39.573 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:31:39.576 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:158 - No palette provided and 4 channels selected; rendering with the default channel palette.


In [18]:
vc = hpv.imgbased_transcriptomics_from_spatialdata(
    sdata.path,
    image_name="clahe",
    labels_name="nucleus_segmentation_mask",
    table_name="table_transcriptomics_preprocessed",
    channels=[0, 1, 2, 3],
    cluster_key="leiden",
    visualize_feature_matrix=True,
    embedding_key="X_umap",
    to_coordinate_system="global_micron",
)

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))

2026-06-14 21:31:54.857 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce ignores zoom unless center is also set.
2026-06-14 21:31:54.860 | INFO     | harpy_vitessce.vitessce_config._image:build_image_layer_config:158 - No palette provided and 4 channels selected; rendering with the default channel palette.
